# RouteHunter

**The app is static.** A CSV file is the single source of truth for the dataset — there is no Contribute/admin pipeline and no in-app way to modify the data. Each CSV you point this at is effectively a version of the app's dataset. To add or correct data, edit the CSV directly and re-run this notebook.

The only thing that happens at runtime beyond reading the CSV is an optional, session-only cache of CASP-predicted routes (Section 4) — it lives in memory for this run and is never written back anywhere.

In [1]:
from routehunter import RouteHunterApp

# This single call is the entire "startup" of the app: load the CSV,
# build the in-memory dataset, done.
app = RouteHunterApp.from_data_dir("rh-data")

print(app.load_report.summary())

[15:31:27] Invalid InChI prefix in generating InChI Key
[15:31:28] Invalid InChI prefix in generating InChI Key
[15:31:28] SMILES Parse Error: extra open parentheses for input: 'C1CC1C(=O)N2CCN(CC2)C(=O)C3=C(C=CC(=C3)CC4=NNC(=O)C5=CC=CC=C54'


Rows processed : 1528
Targets        : 1525
Unique targets : 1395
Unique papers  : 1284
Errors         : 3

First few errors:
  row 348: Could not derive InChIKey for: 'CC1C=C(C)C(NC(CN2CCN(C3CC(CC4C=CC=CC=4)N(C(C4C=C(C(F)(F)F)C=C(C(F)(F)F)C=4)=O)CC3)CC2)=O)=C([*])C=1'
  row 517: Could not derive InChIKey for: 'CC(OC(NC([*])CC(C1C=CC=CC=1)=O)=O)(C)C'
  row 1511: Could not parse SMILES: 'C1CC1C(=O)N2CCN(CC2)C(=O)C3=C(C=CC(=C3)CC4=NNC(=O)C5=CC=CC=C54'


## Review

In [4]:
app.store.stats()

{'n_targets': 1395,
 'n_papers': 1284,
 'n_multi_paper_targets': 106,
 'n_cached_casp_routes': 0,
 'n_predicted_targets': 91546,
 'targets_by_journal': {'Organic Process Research & Development': 1262,
  'Tetrahedron Letters': 2,
  'Synlett': 2,
  'Green Chemistry': 1,
  'Reaction Chemistry & Engineering': 2,
  'Synthesis': 2,
  'Organic Letters': 2,
  'Chemical Science': 2,
  'European Journal of Organic Chemistry': 3,
  'Angewandte Chemie International Edition': 2,
  'The Journal of Organic Chemistry': 2,
  'Journal of the American Chemical Society': 2},
 'targets_by_contributor': {'Dmitry Zankov': 1284}}

In [2]:
print(app.introduction())

RouteHunter -- synthesis route reference lookup (static dataset)

  Search      : give a SMILES, get papers, static CASP-solved tool
                results, and predicted solvability for that molecule.
  Monitor     : browse recently published papers, ranked by predicted
                probability of containing a multi-step synthesis route
                (pre-scored offline; candidates for you to review and
                add to the CSV by hand).
  Predict     : given a SMILES, get predicted solvability probability
                per CASP tool, with a link to that tool.
  Download    : export the dataset for AI/ML training.

This dataset is loaded from a CSV file; there is no in-app way to
modify it. To add or correct data, edit the CSV and reload.


Current dataset:
  Targets                : 1395
  Papers                 : 1284
  Targets w/ >1 route    : 106
  Cached CASP routes     : 0 (session only)
  Predicted targets      : 91546 (awaiting digitalization)
  Papers by journal

## Search

Given a SMILES, return literature papers *and* any CASP-predicted routes cached earlier this session.

Try some molecules with positive search:  
``C#CCOC1=C(C=C(C(=C1)N2C(=O)N3CCCCC3=N2)Cl)Cl``  
``C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O``  
``C1CCC(=C(C1)CC(=O)O)N2C(=O)C=CC(=N2)C3=C4C=CC=CN4N=C3C5=CC=CC=C5``

Try some absent molecules:  
``CC(C)Cc1ccc(cc1)C(C)C(=O)O``

In [ ]:
result = app.search("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.report())

## Monitor

Fetch recent papers, score with a classifier, display ranked. This is **display-only** - nothing here is written into the dataset. If a candidate turns out to be a real route, the way to record it is to add a row to the CSV and reload.

In [ ]:
result  = app.monitor(year_min=1990, year_max=2025)
print(result.message)

In [ ]:
result.to_dataframe()

## Predict

Predict a route computationally. Results are cached in memory for this session (`cache=True` by default) so a later Search this session surfaces them too — but the cache disappears when the notebook restarts; it is never written to the CSV. Uses a stub `CASPEngine` here — swap in a real open-source CASP tool (e.g. AiZynthFinder) via the same `predict_route` interface.

In [ ]:
result = app.predict("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.to_dataframe().to_string())

## Download

Export the dataset (or a filtered slice) as a flat table for ML training. Every row comes from the CSV — Hunter output never appears here since it's never written into the dataset.

In [ ]:
app.download()